In [ ]:
mport pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("D:/Credit-Risk-Scoring-System/data/raw/german_credit_data.csv")
print(df)

In [ ]:
# Basic info
print(df.shape)

In [ ]:
df.head()

In [ ]:
# Column names
df.columns

In [ ]:
# Data types
df.dtypes

In [ ]:
df.info()


In [ ]:
for col in df.columns:
    if df[col].nunique()<20:
        print(df[col].value_counts())
        print('-'*50)

Missing Value Checks

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum()/len(df)*100

In [ ]:
df.isnull().sum().sort_values(ascending=False)

Duplicate check

In [ ]:
df[df.duplicated()].shape# means nothing is duplicated

In [ ]:
df[df.duplicated()]


Statistical Summary

In [ ]:
df.describe()


In [ ]:
df.describe(include='object')

Target Variable Analysis

In [ ]:
# Target distribution
df['Risk'].value_counts()

In [ ]:
sns.countplot(x='Risk', data=df)
plt.title("Credit Risk Distribution")
plt.show()

Numerical Feature Analysis

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
num_cols

In [ ]:
df[num_cols].hist(figsize=(15,10))
plt.show()

Categorical Feature Analysis

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
cat_cols

In [ ]:
for col in cat_cols:
    plt.figure(figsize=(6,3))
    sns.countplot(x=col, data=df)
    plt.xticks(rotation=45)
    plt.title(col)
    plt.show()

Correlation Analysis

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation")
plt.show()

Risk vs Features

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='Risk', y=col, data=df)
    plt.title(f"{col} vs Risk")
    plt.show()

DATA PREPROCESSING AND CLEANING

In [ ]:
#Copy Dataset

df_clean = df.copy()

In [ ]:
#Missing Value Treatment
df_clean.isnull().sum()

In [ ]:
df_clean['Saving accounts'].fillna(df_clean['Saving accounts'].mode()[0], inplace=True)
df_clean['Checking account'].fillna(df_clean['Checking account'].mode()[0], inplace=True)

In [ ]:
df_clean.isnull().sum()

In [ ]:
numeric_cols = ['Age', 'Credit amount', 'Duration']

for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df_clean[col])
    plt.title(f'Boxplot of {col}')
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x=df_clean['Credit amount'])
plt.title("Boxplot of Credit Amount")
plt.show()

In [ ]:
#Outlier Handling(IQR Method)

def remove_outliers_iqr(df,cols):
    for col in cols:
        Q1= df[col].quantile(0.25)
        Q3= df[col].quantile(0.75)
        IQR = Q3-Q1
        lower = Q1 -1.5*IQR
        upper = Q3 + 1.5* IQR
        df[col]= np.clip(df[col],lower,upper)
    return df

df_clean = remove_outliers_iqr(df_clean,num_cols)



In [ ]:
numeric_cols = ['Age', 'Credit amount', 'Duration']

for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df_clean[col])
    plt.title(f'Boxplot of {col}')
    plt.show()


In [ ]:
df_clean['Risk'].unique()

In [ ]:
#target variable encoding

df_clean['Risk'] = (
    df_clean['Risk']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '', regex=True)
    .map({'good': 0, 'bad': 1})
)

In [ ]:
print(df_clean.dtypes)
print(df_clean[['Sex','Housing','Purpose','Saving accounts','Checking account']].head())

In [ ]:
for col in ['Sex','Housing','Purpose','Saving accounts','Checking account']:
    df_clean[col] = df_clean[col].astype(str).str.strip()

In [ ]:
# Check for problematic values in categorical columns
categorical_cols = ['Sex','Housing','Purpose','Saving accounts','Checking account']

for col in categorical_cols:
    bad_rows = df_clean[df_clean[col].apply(lambda x: isinstance(x, (list, tuple, np.ndarray)))]
    if not bad_rows.empty:
        print(f"Column '{col}' has array-like values at rows:")
        print(bad_rows[[col]].head())


for col in categorical_cols:
    df_clean[col] = df_clean[col].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)
    df_clean[col] = df_clean[col].astype(str).str.strip()

df_clean.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# -----------------------------
# STEP 1: One-Hot Encoding
# -----------------------------
onehot_cols = ['Sex', 'Housing', 'Purpose']

onehot_encoder = OneHotEncoder(drop='first', sparse_output=False)
onehot_encoded = onehot_encoder.fit_transform(df_clean[onehot_cols])

onehot_df = pd.DataFrame(
    onehot_encoded,
    columns=onehot_encoder.get_feature_names_out(onehot_cols),
    index=df_clean.index
)

df_clean_encoded = pd.concat(
    [df_clean.drop(columns=onehot_cols), onehot_df],
    axis=1
)

In [ ]:
# STEP 2: Ordinal Encoding
# -----------------------------
saving_map = {'little': 0, 'moderate': 1, 'quite rich': 2, 'rich': 3}
checking_map = {'little': 0, 'moderate': 1, 'rich': 2}

df_clean_encoded['Saving accounts'] = df_clean_encoded['Saving accounts'].replace(saving_map)
df_clean_encoded['Checking account'] = df_clean_encoded['Checking account'].replace(checking_map)

In [ ]:
# -----------------------------
# STEP 3: Scaling
# -----------------------------
numeric_cols = ['Age', 'Credit amount', 'Duration']

scaler = StandardScaler()
df_clean_encoded[numeric_cols] = scaler.fit_transform(df_clean_encoded[numeric_cols])


In [ ]:
# -----------------------------
# FINAL DATASET
# -----------------------------
print(df_clean_encoded.head())

In [ ]:
df_clean['Checking account'].unique()

In [ ]:
type("rich")   # <class 'str'>

In [ ]:
for col in ['Saving accounts','Checking account']:
    print(col, df_clean[col].apply(type).unique())

In [ ]:
'''import numpy as np

# Identify categorical columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns

# Clean all categorical columns
for col in categorical_cols:
    df_clean[col] = df_clean[col].apply(
        lambda x: str(x[0]).strip() if isinstance(x, (list, np.ndarray)) else str(x).strip()
    )'''

In [ ]:
''''import numpy as np

# Identify categorical columns automatically
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print("Categorical columns:", categorical_cols)

# Clean each categorical column
for col in categorical_cols:
    df_clean[col] = df_clean[col].apply(
        lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x
    )
    df_clean[col] = df_clean[col].astype(str).str.strip()'''

In [ ]:
'''# Define mappings
saving_map = {'little': 0, 'moderate': 1, 'quite rich': 2, 'rich': 3}
checking_map = {'little': 0, 'moderate': 1, 'rich': 2}

# Apply mappings directly
df_clean['Saving accounts_encoded'] = df_clean['Saving accounts'].replace(saving_map)
df_clean['Checking account_encoded'] = df_clean['Checking account'].replace(checking_map)

# Drop old columns if you want
df_clean_encoded = df_clean.drop(columns=['Saving accounts','Checking account'])

print(df_clean_encoded.head())'''

In [ ]:
'''#Scaling numerical features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = ['Age', 'Credit amount', 'Duration']

# Fit the scaler on training data
scaler.fit(df_clean[numeric_cols])

# Transform the data (mean=0, std=1)
df_clean[numeric_cols] = scaler.transform(df_clean[numeric_cols])

print(df_clean.head())'''

Feature/Target Split

In [ ]:
x = df_clean_encoded.drop(['Risk', 'Unnamed: 0'], axis=1)
y = df_clean_encoded['Risk']

x.head(5)
x.dtypes

In [ ]:
y.value_counts()

In [ ]:
y.shape
y.dtypes
y.head(5)

Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Encode first
x = df_clean_encoded.drop(['Risk', 'Unnamed: 0'], axis=1)
y = df_clean_encoded['Risk']

# Then split
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)



In [ ]:
print(y_train.unique())
print(y_train.dtype)

In [ ]:
print(X_train.dtypes)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'   # 👈 automatic balancing
)

log_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve,accuracy_score,precision_score,recall_score,f1_score

# Predictions
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:,1]

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_prob))

In [ ]:
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure()
plt.plot(fpr, tpr, label="Logistic Regression")
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Credit Risk Model")
plt.legend()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve,accuracy_score,precision_score,recall_score,f1_score


# Model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',   # 🔥 handles imbalance properly
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:,1]

# Evaluation
print("🔹 Random Forest Results")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_prob_rf))

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve,accuracy_score,precision_score,recall_score,f1_score


# Handle imbalance ratio
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,  # 🔥 imbalance handling
    random_state=42
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:,1]

# Evaluation
print("🔹 XGBoost Results")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_prob_xgb))

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score, f1_score
import pandas as pd

# Logistic Regression metrics
log_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "Recall(BAD)": recall_score(y_test, y_pred),
    "Precision(BAD)": precision_score(y_test, y_pred),
    "F1-Score": f1_score(y_test, y_pred)
}

# Random Forest metrics
rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
    "Recall(BAD)": recall_score(y_test, y_pred_rf),
    "Precision(BAD)": precision_score(y_test, y_pred_rf),
    "F1-Score": f1_score(y_test, y_pred_rf)
}

# XGBoost metrics
xgb_metrics = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred_xgb),
    "ROC-AUC": roc_auc_score(y_test, y_prob_xgb),
    "Recall(BAD)": recall_score(y_test, y_pred_xgb),
    "Precision(BAD)": precision_score(y_test, y_pred_xgb),
    "F1-Score": f1_score(y_test, y_pred_xgb)
}

# Create comparison table
comparison_df = pd.DataFrame([log_metrics, rf_metrics, xgb_metrics])

# Sort by ROC-AUC (main business metric)
comparison_df = comparison_df.sort_values(by="ROC-AUC", ascending=False)

comparison_df

In [ ]:
import joblib

best_model = xgb_model  # example
joblib.dump(best_model, "D:/Credit-Risk-Scoring-System/model/credit_risk_model.pkl")